# API Test Case Generator
**Stack:** LangChain + LangGraph + Ollama (llama3.2)  
**Domain:** QA / Banking / Insurance REST APIs  

**Agent flow:**
```
API Spec Input (endpoint, method, params, auth, business rules)
      ↓
[API Analyzer Node]       — parses endpoint contract
      ↓
[Test Case Builder Node]  — generates functional, negative, security, perf test cases
      ↓
[Curl Generator Node]     — writes ready-to-run curl / Python requests snippets
      ↓
Complete API Test Suite output
```

In [1]:
%pip install -q langchain langchain-ollama langgraph

Note: you may need to restart the kernel to use updated packages.


In [1]:
import json
from typing import TypedDict
from IPython.display import display, Markdown
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
print('Imports OK')

Imports OK


In [2]:
import urllib.request, json as _j
try:
    with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3) as r:
        models = [m["name"] for m in _j.loads(r.read()).get("models", [])]
        print("Ollama running. Models:", models)
        if not any("llama3.2" in m for m in models):
            print("WARNING: llama3.2 not found. Run: ollama pull llama3.2")
except Exception as e:
    print("Ollama not reachable:", e)
    print("Start Ollama from system tray or run: ollama serve")


Ollama running. Models: ['llama3.2:latest']


In [3]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    base_url="http://localhost:11434",
    temperature=0.3,
)
print("LLM ready:", llm.model)


LLM ready: llama3.2


## State Definition

In [4]:
class APITCState(TypedDict):
    api_spec: str           # input: endpoint spec
    api_analysis: str       # parsed contract details
    test_cases: str         # functional + negative + security test cases
    curl_snippets: str      # ready-to-run request examples
    final_suite: str        # combined test suite


## Node 1 — API Analyzer

In [5]:
api_analyze_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a senior API QA engineer specializing in banking and insurance REST APIs. "
     "Analyze the API specification and extract:\n"
     "1. Endpoint and HTTP method\n"
     "2. Request headers (auth type, content-type)\n"
     "3. Request body/params — mandatory vs optional, data types, constraints\n"
     "4. Expected response codes (2xx, 4xx, 5xx) and response schema\n"
     "5. Business rules and validations\n"
     "6. Security considerations (auth, PII fields, rate limiting)\n"
     "7. Idempotency / retry behavior"),
    ("human", "API Specification:\n{api_spec}")
])

api_analyze_chain = api_analyze_prompt | llm | StrOutputParser()

def api_analyzer_node(state: APITCState) -> APITCState:
    print("[Node 1] Analyzing API specification...")
    analysis = api_analyze_chain.invoke({"api_spec": state["api_spec"]})
    return {**state, "api_analysis": analysis}


## Node 2 — Test Case Builder

In [6]:
tcb_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert API test case designer for financial systems. "
     "Generate a complete API test suite with the following test types:\n\n"
     "FUNCTIONAL TESTS (Happy Path):\n"
     "- Valid request with all mandatory fields\n"
     "- Valid request with optional fields\n\n"
     "NEGATIVE TESTS:\n"
     "- Missing mandatory fields\n"
     "- Invalid data types\n"
     "- Out-of-range values\n"
     "- Boundary value tests\n\n"
     "AUTHENTICATION & SECURITY TESTS:\n"
     "- Missing token / expired token\n"
     "- Invalid token / wrong scope\n"
     "- SQL injection in params\n"
     "- Excessive data in payload\n\n"
     "PERFORMANCE INDICATORS:\n"
     "- SLA threshold for response time\n"
     "- Concurrent request behavior\n\n"
     "Format each test case as:\n"
     "TC_ID | Type | Description | Input | Expected HTTP Code | Expected Response"),
    ("human",
     "API Spec:\n{api_spec}\n\n"
     "Analysis:\n{api_analysis}")
])

tcb_chain = tcb_prompt | llm | StrOutputParser()

def test_case_builder_node(state: APITCState) -> APITCState:
    print("[Node 2] Building API test cases...")
    tcs = tcb_chain.invoke({
        "api_spec": state["api_spec"],
        "api_analysis": state["api_analysis"]
    })
    return {**state, "test_cases": tcs}


## Node 3 — Curl & Python Snippet Generator

In [7]:
curl_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an API automation engineer. Generate ready-to-run examples for:\n"
     "1. A valid happy-path curl command\n"
     "2. A negative test curl command (e.g. missing field)\n"
     "3. An auth failure curl command (no/bad token)\n"
     "4. The equivalent Python requests snippet for the happy-path test\n\n"
     "Use realistic placeholder values. "
     "Add a comment above each snippet explaining what it tests."),
    ("human",
     "API Spec:\n{api_spec}\n\n"
     "Test Cases (for context):\n{test_cases}")
])

curl_chain = curl_prompt | llm | StrOutputParser()

def curl_generator_node(state: APITCState) -> APITCState:
    print("[Node 3] Generating curl and Python snippets...")
    snippets = curl_chain.invoke({
        "api_spec": state["api_spec"],
        "test_cases": state["test_cases"]
    })
    suite = (
        f"# API TEST SUITE\n\n"
        f"## API Contract Analysis\n{state['api_analysis']}\n\n"
        f"## Test Cases\n{state['test_cases']}\n\n"
        f"## Ready-to-Run Snippets\n{snippets}"
    )
    return {**state, "curl_snippets": snippets, "final_suite": suite}


## Build the LangGraph

In [8]:
graph = StateGraph(APITCState)
graph.add_node("api_analyzer",      api_analyzer_node)
graph.add_node("test_case_builder", test_case_builder_node)
graph.add_node("curl_generator",    curl_generator_node)

graph.set_entry_point("api_analyzer")
graph.add_edge("api_analyzer",      "test_case_builder")
graph.add_edge("test_case_builder", "curl_generator")
graph.add_edge("curl_generator",    END)

app = graph.compile()
print("Graph compiled. Flow: api_analyzer → test_case_builder → curl_generator → END")


Graph compiled. Flow: api_analyzer → test_case_builder → curl_generator → END


## Run the Agent
Paste any REST API spec below. Banking examples provided.

In [9]:
api_spec = """
API Name: Fund Transfer API
Endpoint: POST /api/v2/accounts/transfer
Auth: Bearer Token (OAuth 2.0, scope: transfers:write)
Content-Type: application/json

Request Body:
{
  "from_account": string (mandatory) — source account number (10 digits)
  "to_account":   string (mandatory) — destination account number (10 digits)
  "amount":       number (mandatory) — transfer amount, min: 0.01, max: 10000.00
  "currency":     string (mandatory) — ISO 4217 code e.g. USD, GBP, SGD
  "reference":    string (optional)  — customer reference note, max 100 chars
  "transfer_type":string (mandatory) — INTERNAL or EXTERNAL
}

Business Rules:
- from_account must belong to the authenticated user
- Insufficient balance must return 422 with error code INSUFFICIENT_FUNDS
- Daily transfer limit is $10,000 cumulative
- EXTERNAL transfers require additional 2FA verification header: X-OTP-Token
- Duplicate requests within 60 seconds must be rejected (idempotency key)

Success Response: 201 Created
{
  "transaction_id": "TXN-20260609-00123",
  "status": "COMPLETED",
  "timestamp": "2026-06-09T10:30:00Z"
}

Error Codes: 400 Bad Request, 401 Unauthorized, 403 Forbidden,
             409 Conflict (duplicate), 422 Unprocessable Entity, 500 Internal Server Error
"""

# ── Insurance API example (uncomment to use) ────────────────────
# api_spec = """
# API Name: Claims Submission API
# Endpoint: POST /api/v1/claims/submit
# Auth: Bearer Token (scope: claims:write)
# Request Body:
#   policy_number: string (mandatory)
#   incident_date: date   (mandatory, cannot be future date)
#   claim_type:    string (mandatory) — MOTOR, PROPERTY, HEALTH
#   damage_amount: number (mandatory) — min 100, max 500000
#   description:   string (mandatory) — max 500 chars
#   documents:     array  (optional)  — list of base64 encoded files
# Business Rules:
# - Policy must be active and not expired
# - Incident date cannot be more than 30 days in the past
# - MOTOR claims require a police report number
# Success: 201 with claim_id
# """

initial_state: APITCState = {
    "api_spec": api_spec.strip(),
    "api_analysis": "",
    "test_cases": "",
    "curl_snippets": "",
    "final_suite": "",
}

print("Running API Test Case Generator Agent...")
result = app.invoke(initial_state)
print("Agent completed.")


Running API Test Case Generator Agent...
[Node 1] Analyzing API specification...
[Node 2] Building API test cases...
[Node 3] Generating curl and Python snippets...
Agent completed.


## Results

In [10]:
print(result['api_analysis'])

Here's the extracted information:

**1. Endpoint and HTTP method**

* Endpoint: `POST /api/v2/accounts/transfer`
* HTTP Method: `POST`

**2. Request headers (auth type, content-type)**

* Auth Type: Bearer Token (OAuth 2.0, scope: transfers:write)
* Content-Type: `application/json`

**3. Request body/params**

| Field | Mandatory | Data Type | Constraints |
| --- | --- | --- | --- |
| from_account | Yes | string (10 digits) | - |
| to_account | Yes | string (10 digits) | - |
| amount | Yes | number | min: 0.01, max: 10000.00 |
| currency | Yes | string (ISO 4217 code) | - |
| reference | No | string (max 100 chars) | - |
| transfer_type | Yes | string (INTERNAL or EXTERNAL) | - |

**4. Expected response codes**

* 2xx: `201 Created` (Success)
* 4xx:
	+ 400 Bad Request
	+ 401 Unauthorized
	+ 403 Forbidden
* 5xx: `500 Internal Server Error`

**5. Business rules and validations**

* From account must belong to the authenticated user.
* Insufficient balance returns `422 Unprocessable Entit

In [11]:
print(result['test_cases'])

Here's a comprehensive API test suite for the Fund Transfer API:

**FUNCTIONAL TESTS (Happy Path)**

1. **Valid request with all mandatory fields**
	* TC_ID: FT-001
	* Type: Functional Test
	* Description: Verify successful transfer of funds with valid input data.
	* Input:
		+ from_account: `1234567890`
		+ to_account: `9876543210`
		+ amount: `100.00`
		+ currency: `USD`
		+ transfer_type: `INTERNAL`
	* Expected HTTP Code: 201 Created
	* Expected Response:
		+ transaction_id: `TXN-20260609-00123`
		+ status: `COMPLETED`
		+ timestamp: `2026-06-09T10:30:00Z`
2. **Valid request with optional fields**
	* TC_ID: FT-002
	* Type: Functional Test
	* Description: Verify successful transfer of funds with valid input data and optional fields.
	* Input:
		+ from_account: `1234567890`
		+ to_account: `9876543210`
		+ amount: `100.00`
		+ currency: `USD`
		+ transfer_type: `INTERNAL`
		+ reference: `Test Reference Note`
	* Expected HTTP Code: 201 Created
	* Expected Response:
		+ transaction_id: 

In [12]:
print(result['curl_snippets'])

Here are the ready-to-run examples for each test case:

**Happy Path: Valid Request with All Mandatory Fields**

```bash
curl -X POST \
  https://api.example.com/api/v2/accounts/transfer \
  -H 'Authorization: Bearer YOUR_BEARER_TOKEN' \
  -H 'Content-Type: application/json' \
  -d '{"from_account": "1234567890", "to_account": "9876543210", "amount": "100.00", "currency": "USD", "transfer_type": "INTERNAL"}'
```

**Negative Test: Missing Mandatory Fields**

```bash
curl -X POST \
  https://api.example.com/api/v2/accounts/transfer \
  -H 'Authorization: Bearer YOUR_BEARER_TOKEN' \
  -H 'Content-Type: application/json' \
  -d '{"from_account": "1234567890", "to_account": "9876543210"}'
```

**Auth Failure Test: Missing Token**

```bash
curl -X POST \
  https://api.example.com/api/v2/accounts/transfer \
  -H 'Content-Type: application/json' \
  -d '{"from_account": "1234567890", "to_account": "9876543210", "amount": "100.00", "currency": "USD", "transfer_type": "INTERNAL"}'
```

**Equival

In [13]:
display(Markdown(result['final_suite']))

# API TEST SUITE

## API Contract Analysis
Here's the extracted information:

**1. Endpoint and HTTP method**

* Endpoint: `POST /api/v2/accounts/transfer`
* HTTP Method: `POST`

**2. Request headers (auth type, content-type)**

* Auth Type: Bearer Token (OAuth 2.0, scope: transfers:write)
* Content-Type: `application/json`

**3. Request body/params**

| Field | Mandatory | Data Type | Constraints |
| --- | --- | --- | --- |
| from_account | Yes | string (10 digits) | - |
| to_account | Yes | string (10 digits) | - |
| amount | Yes | number | min: 0.01, max: 10000.00 |
| currency | Yes | string (ISO 4217 code) | - |
| reference | No | string (max 100 chars) | - |
| transfer_type | Yes | string (INTERNAL or EXTERNAL) | - |

**4. Expected response codes**

* 2xx: `201 Created` (Success)
* 4xx:
	+ 400 Bad Request
	+ 401 Unauthorized
	+ 403 Forbidden
* 5xx: `500 Internal Server Error`

**5. Business rules and validations**

* From account must belong to the authenticated user.
* Insufficient balance returns `422 Unprocessable Entity` with error code `INSUFFICIENT_FUNDS`.
* Daily transfer limit is $10,000 cumulative.
* EXTERNAL transfers require additional 2FA verification header: `X-OTP-Token`.
* Duplicate requests within 60 seconds are rejected (idempotency key).

**6. Security considerations**

* Auth Token (OAuth 2.0) with scope: `transfers:write` is used for authentication.
* PII fields (e.g., account numbers, amounts) should be masked or encrypted in transit and at rest.
* Rate limiting can be implemented to prevent abuse of the API.

**7. Idempotency / retry behavior**

* Duplicate requests within 60 seconds are rejected due to idempotency key.
* No explicit retry mechanism is mentioned; however, a retry policy could be implemented for transient errors (e.g., network issues).

## Test Cases
Here's a comprehensive API test suite for the Fund Transfer API:

**FUNCTIONAL TESTS (Happy Path)**

1. **Valid request with all mandatory fields**
	* TC_ID: FT-001
	* Type: Functional Test
	* Description: Verify successful transfer of funds with valid input data.
	* Input:
		+ from_account: `1234567890`
		+ to_account: `9876543210`
		+ amount: `100.00`
		+ currency: `USD`
		+ transfer_type: `INTERNAL`
	* Expected HTTP Code: 201 Created
	* Expected Response:
		+ transaction_id: `TXN-20260609-00123`
		+ status: `COMPLETED`
		+ timestamp: `2026-06-09T10:30:00Z`
2. **Valid request with optional fields**
	* TC_ID: FT-002
	* Type: Functional Test
	* Description: Verify successful transfer of funds with valid input data and optional fields.
	* Input:
		+ from_account: `1234567890`
		+ to_account: `9876543210`
		+ amount: `100.00`
		+ currency: `USD`
		+ transfer_type: `INTERNAL`
		+ reference: `Test Reference Note`
	* Expected HTTP Code: 201 Created
	* Expected Response:
		+ transaction_id: `TXN-20260609-00123`
		+ status: `COMPLETED`
		+ timestamp: `2026-06-09T10:30:00Z`

**NEGATIVE TESTS**

1. **Missing mandatory fields**
	* TC_ID: FT-003
	* Type: Negative Test
	* Description: Verify error response when missing required fields.
	* Input:
		+ from_account: `1234567890`
		+ to_account: `9876543210`
		+ amount: `100.00`
		+ currency: `USD`
		+ transfer_type: `INTERNAL` (missing)
	* Expected HTTP Code: 400 Bad Request
	* Expected Response:
		+ error_code: `MISSING_FIELDS`
2. **Invalid data types**
	* TC_ID: FT-004
	* Type: Negative Test
	* Description: Verify error response when invalid data type is used.
	* Input:
		+ from_account: `1234567890abc`
		+ to_account: `9876543210`
		+ amount: `100.00`
		+ currency: `USD` (invalid ISO code)
		+ transfer_type: `INTERNAL`
	* Expected HTTP Code: 400 Bad Request
	* Expected Response:
		+ error_code: `INVALID_DATA_TYPE`
3. **Out-of-range values**
	* TC_ID: FT-005
	* Type: Negative Test
	* Description: Verify error response when out-of-range value is used.
	* Input:
		+ from_account: `1234567890`
		+ to_account: `9876543210`
		+ amount: `-100.00` (less than min)
		+ currency: `USD`
		+ transfer_type: `INTERNAL`
	* Expected HTTP Code: 400 Bad Request
	* Expected Response:
		+ error_code: `OUT_OF_RANGE_VALUE`
4. **Boundary value tests**
	* TC_ID: FT-006
	* Type: Negative Test
	* Description: Verify successful transfer of funds with boundary values.
	* Input:
		+ from_account: `1234567890`
		+ to_account: `9876543210`
		+ amount: `0.01` (min value)
		+ currency: `USD`
		+ transfer_type: `INTERNAL`
	* Expected HTTP Code: 201 Created
	* Expected Response:
		+ transaction_id: `TXN-20260609-00123`
		+ status: `COMPLETED`
		+ timestamp: `2026-06-09T10:30:00Z`

**AUTHENTICATION & SECURITY TESTS**

1. **Missing token / expired token**
	* TC_ID: AT-001
	* Type: Authentication Test
	* Description: Verify error response when missing or expired token is used.
	* Input:
		+ from_account: `1234567890`
		+ to_account: `9876543210`
		+ amount: `100.00`
		+ currency: `USD`
		+ transfer_type: `INTERNAL` (missing token)
	* Expected HTTP Code: 401 Unauthorized
	* Expected Response:
		+ error_code: `INVALID_TOKEN`
2. **Invalid token / wrong scope**
	* TC_ID: AT-002
	* Type: Authentication Test
	* Description: Verify error response when invalid or incorrect token is used.
	* Input:
		+ from_account: `1234567890`
		+ to_account: `9876543210`
		+ amount: `100.00`
		+ currency: `USD`
		+ transfer_type: `INTERNAL` (wrong scope)
	* Expected HTTP Code: 401 Unauthorized
	* Expected Response:
		+ error_code: `INVALID_TOKEN`
3. **SQL injection in params**
	* TC_ID: AT-003
	* Type: Security Test
	* Description: Verify error response when SQL injection is attempted.
	* Input:
		+ from_account: `1234567890`
		+ to_account: `9876543210`
		+ amount: `100.00`
		+ currency: `USD` (SQL injection attempt)
		+ transfer_type: `INTERNAL`
	* Expected HTTP Code: 400 Bad Request
	* Expected Response:
		+ error_code: `SQL_INJECTION`
4. **Excessive data in payload**
	* TC_ID: AT-004
	* Type: Security Test
	* Description: Verify error response when excessive data is sent.
	* Input:
		+ from_account: `1234567890`
		+ to_account: `9876543210`
		+ amount: `100.00`
		+ currency: `USD` (excessive data)
		+ transfer_type: `INTERNAL`
	* Expected HTTP Code: 400 Bad Request
	* Expected Response:
		+ error_code: `EXCESSIVE_DATA`

**PERFORMANCE INDICATORS**

1. **SLA threshold for response time**
	* TC_ID: PI-001
	* Type: Performance Indicator
	* Description: Verify that the API responds within the SLA threshold.
	* Input:
		+ from_account: `1234567890`
		+ to_account: `9876543210`
		+ amount: `100.00`
		+ currency: `USD`
		+ transfer_type: `INTERNAL` (no delay)
	* Expected HTTP Code: 201 Created
	* Expected Response:
		+ transaction_id: `TXN-20260609-00123`
		+ status: `COMPLETED`
		+ timestamp: `2026-06-09T10:30:00Z` (within SLA threshold)
2. **Concurrent request behavior**
	* TC_ID: PI-002
	* Type: Performance Indicator
	* Description: Verify that the API handles concurrent requests correctly.
	* Input:
		+ from_account: `1234567890`
		+ to_account: `9876543210`
		+ amount: `100.00`
		+ currency: `USD`
		+ transfer_type: `INTERNAL` (multiple concurrent requests)
	* Expected HTTP Code: 201 Created
	* Expected Response:
		+ transaction_id: `TXN-20260609-00123`, `TXN-20260609-00124` (correct handling of concurrent requests)

Note that this is not an exhaustive list, and additional test cases may be required to cover all possible scenarios.

## Ready-to-Run Snippets
Here are the ready-to-run examples for each test case:

**Happy Path: Valid Request with All Mandatory Fields**

```bash
curl -X POST \
  https://api.example.com/api/v2/accounts/transfer \
  -H 'Authorization: Bearer YOUR_BEARER_TOKEN' \
  -H 'Content-Type: application/json' \
  -d '{"from_account": "1234567890", "to_account": "9876543210", "amount": "100.00", "currency": "USD", "transfer_type": "INTERNAL"}'
```

**Negative Test: Missing Mandatory Fields**

```bash
curl -X POST \
  https://api.example.com/api/v2/accounts/transfer \
  -H 'Authorization: Bearer YOUR_BEARER_TOKEN' \
  -H 'Content-Type: application/json' \
  -d '{"from_account": "1234567890", "to_account": "9876543210"}'
```

**Auth Failure Test: Missing Token**

```bash
curl -X POST \
  https://api.example.com/api/v2/accounts/transfer \
  -H 'Content-Type: application/json' \
  -d '{"from_account": "1234567890", "to_account": "9876543210", "amount": "100.00", "currency": "USD", "transfer_type": "INTERNAL"}'
```

**Equivalent Python Requests Snippet for Happy Path**

```python
import requests

bearer_token = 'YOUR_BEARER_TOKEN'

response = requests.post(
    'https://api.example.com/api/v2/accounts/transfer',
    headers={'Authorization': f'Bearer {bearer_token}', 'Content-Type': 'application/json'},
    json={
        'from_account': '1234567890',
        'to_account': '9876543210',
        'amount': 100.00,
        'currency': 'USD',
        'transfer_type': 'INTERNAL'
    }
)

print(response.json())
```

Note: Replace `YOUR_BEARER_TOKEN` with your actual bearer token.

Also, make sure to handle any exceptions or errors that may occur during the API call, such as network errors or invalid responses.